# Day 047 Solution — Statistics for AI Engineers

Demonstrates: extended descriptive stats, normality testing, correlation with p-values, group comparison (t-test + Cohen's d), StatsReport class, and a saved matplotlib figure.
All data generated in-cell — no external files required.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

def make_sample_data(n: int = 100, seed: int = 42) -> pd.DataFrame:
    """Return a reproducible multi-column dataset for statistics exercises."""
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'normal_col': rng.standard_normal(n).round(3),
        'skewed_col': rng.exponential(2, n).round(3),
        'score_a':    (50 + rng.standard_normal(n) * 10).round(1),
        'score_b':    (70 + rng.standard_normal(n) * 10).round(1),
    })


def describe_distribution(series: pd.Series) -> dict:
    s   = series.dropna()
    q25 = float(s.quantile(0.25))
    q75 = float(s.quantile(0.75))
    return {
        'count':    int(len(s)),
        'mean':     round(float(s.mean()), 4),
        'median':   round(float(s.median()), 4),
        'std':      round(float(s.std(ddof=1)), 4),
        'sem':      round(float(s.sem()), 4),
        'min':      round(float(s.min()), 4),
        'max':      round(float(s.max()), 4),
        'q25':      round(q25, 4),
        'q75':      round(q75, 4),
        'iqr':      round(q75 - q25, 4),
        'skewness': round(float(s.skew()), 4),
        'kurtosis': round(float(s.kurt()), 4),
    }


def test_normality(series: pd.Series, alpha: float = 0.05) -> dict:
    s      = series.dropna()
    stat, p = stats.shapiro(s)
    return {
        'n':          len(s),
        'statistic':  round(float(stat), 4),
        'p_value':    round(float(p), 6),
        'is_normal':  bool(p > alpha),
        'alpha':      alpha,
    }


def correlation_with_pvalue(x: pd.Series, y: pd.Series,
                             method: str = 'pearson') -> dict:
    mask  = x.notna() & y.notna()
    x_c, y_c = x[mask], y[mask]
    if method == 'pearson':
        r, p = stats.pearsonr(x_c, y_c)
    elif method == 'spearman':
        r, p = stats.spearmanr(x_c, y_c)
    else:
        raise ValueError(f"method must be 'pearson' or 'spearman', got {method!r}")
    return {
        'method':         method,
        'n':              len(x_c),
        'r':              round(float(r), 4),
        'p_value':        round(float(p), 6),
        'is_significant': bool(p < 0.05),
    }


def compare_groups(a: pd.Series, b: pd.Series,
                   alpha: float = 0.05) -> dict:
    a_c, b_c  = a.dropna(), b.dropna()
    t, p       = stats.ttest_ind(a_c, b_c)
    n_a, n_b   = len(a_c), len(b_c)
    std_a      = float(a_c.std(ddof=1))
    std_b      = float(b_c.std(ddof=1))
    pooled_var = ((n_a - 1) * std_a**2 + (n_b - 1) * std_b**2) / (n_a + n_b - 2)
    pooled     = np.sqrt(pooled_var) if pooled_var > 0 else 0.0
    d          = (float(a_c.mean()) - float(b_c.mean())) / pooled if pooled > 0 else 0.0
    sig        = bool(p < alpha)
    return {
        'n_a':            n_a,
        'n_b':            n_b,
        'mean_a':         round(float(a_c.mean()), 4),
        'mean_b':         round(float(b_c.mean()), 4),
        'statistic':      round(float(t), 4),
        'p_value':        round(float(p), 6),
        'is_significant': sig,
        'cohens_d':       round(d, 4),
        'conclusion':     'different' if sig else 'not_different',
    }


class StatsReport:
    def __init__(self):
        self._df      = None
        self._columns = None

    def load(self, df: pd.DataFrame,
             columns: list | None = None) -> 'StatsReport':
        self._df      = df.copy()
        num_cols      = df.select_dtypes(include='number').columns.tolist()
        self._columns = columns if columns is not None else num_cols
        return self

    def report(self) -> dict:
        result = {}
        for col in self._columns:
            if col not in self._df.columns:
                continue
            s = self._df[col].dropna()
            if len(s) < 3:
                continue
            result[col] = {
                'distribution': describe_distribution(s),
                'normality':    test_normality(s),
            }
        return result

## Step 1 — Descriptive Statistics

In [ ]:
df = make_sample_data(n=100, seed=42)

for col in df.columns:
    d = describe_distribution(df[col])
    print(f'{col}:')
    print(f'  mean={d["mean"]:.3f}  median={d["median"]:.3f}  std={d["std"]:.3f}')
    print(f'  IQR={d["iqr"]:.3f}  skew={d["skewness"]:.3f}  kurt={d["kurtosis"]:.3f}')
    print(f'  SEM={d["sem"]:.4f}')

d_norm = describe_distribution(df['normal_col'])
d_skew = describe_distribution(df['skewed_col'])
assert d_norm['count'] == 100
assert d_skew['skewness'] > 0, 'exponential column should be right-skewed'

## Step 2 — Normality Testing

In [ ]:
for col in df.columns:
    n = test_normality(df[col])
    status = 'normal' if n['is_normal'] else 'NOT normal'
    print(f'{col}: {status}  (W={n["statistic"]:.4f}, p={n["p_value"]:.4f})')

assert test_normality(df['normal_col'])['is_normal']  in (True, False)  # may vary
assert test_normality(df['skewed_col'])['is_normal'] is False

## Step 3 — Correlation with p-value

In [ ]:
# Create a correlated pair: score_c ≈ 0.7 * score_a + noise
rng2   = np.random.default_rng(99)
score_c = df['score_a'] * 0.7 + pd.Series(rng2.standard_normal(100) * 8)

r_pearson = correlation_with_pvalue(df['score_a'], score_c)
r_spearman = correlation_with_pvalue(df['score_a'], score_c, method='spearman')
r_indep   = correlation_with_pvalue(df['score_a'], df['score_b'])

print(f'score_a vs score_c  (Pearson):  r={r_pearson["r"]:.3f}  p={r_pearson["p_value"]:.4f}  sig={r_pearson["is_significant"]}')
print(f'score_a vs score_c  (Spearman): r={r_spearman["r"]:.3f}  p={r_spearman["p_value"]:.4f}  sig={r_spearman["is_significant"]}')
print(f'score_a vs score_b  (Pearson):  r={r_indep["r"]:.3f}  p={r_indep["p_value"]:.4f}  sig={r_indep["is_significant"]}')

assert r_pearson['is_significant'] is True,  'correlated pair should be significant'
assert abs(r_pearson['r']) > 0.4,            'correlation should be moderate+'

## Step 4 — Group Comparison

In [ ]:
comparison = compare_groups(df['score_a'], df['score_b'])

print('=== Group Comparison ===')
print(f'Group A: n={comparison["n_a"]}  mean={comparison["mean_a"]:.2f}')
print(f'Group B: n={comparison["n_b"]}  mean={comparison["mean_b"]:.2f}')
print(f'  t = {comparison["statistic"]:.3f}   p = {comparison["p_value"]:.2e}')
print(f"  Cohen's d = {comparison['cohens_d']:.3f}  → conclusion: {comparison['conclusion']}")

assert comparison['conclusion'] == 'different'
assert abs(comparison['cohens_d']) > 1.0

## Step 5 — StatsReport

In [ ]:
report = StatsReport().load(df).report()

print('=== Stats Report ===')
for col, entry in report.items():
    d = entry['distribution']
    n = entry['normality']
    norm_str = 'normal' if n['is_normal'] else 'NOT normal'
    print(f'{col}: mean={d["mean"]:.2f} skew={d["skewness"]:.2f} [{norm_str}]')

assert len(report) == 4
assert all('distribution' in v and 'normality' in v for v in report.values())

## Step 6 — Save Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Histogram: normal column
axes[0].hist(df['normal_col'], bins=20, edgecolor='white', color='steelblue')
axes[0].axvline(df['normal_col'].mean(), color='red', linestyle='--', label='mean')
axes[0].set_title('normal_col')
axes[0].set_xlabel('value')
axes[0].legend()

# Histogram: skewed column
axes[1].hist(df['skewed_col'], bins=20, edgecolor='white', color='orange')
axes[1].axvline(df['skewed_col'].mean(),   color='red',  linestyle='--', label='mean')
axes[1].axvline(df['skewed_col'].median(), color='blue', linestyle='--', label='median')
axes[1].set_title('skewed_col (right-skewed)')
axes[1].set_xlabel('value')
axes[1].legend()

# Box plot: score_a vs score_b
axes[2].boxplot([df['score_a'].values, df['score_b'].values],
                labels=['Score A', 'Score B'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[2].set_title(f"Group Comparison (d={comparison['cohens_d']:.2f})")
axes[2].set_ylabel('score')

plt.tight_layout()
fig.savefig('stats_report.png', bbox_inches='tight', dpi=100)
plt.close('all')
print('Chart saved: stats_report.png')

import os
assert os.path.exists('stats_report.png') and os.path.getsize('stats_report.png') > 1000

print('\nStatistics for AI Engineers complete!')